In [21]:
import pandas as pd
import requests


### Описание основных атрибутов

Для дальнейшего анализа отобраны атрибуты, которые непосредственно характеризуют стоимость аренды, географическое положение объектов и основные параметры жилого фонда.

| Атрибут | Физический смысл | Тип шкалы | Роль в анализе |
|---|---|---|---|
| `cluster_id` | Уникальный идентификатор географического кластера | Номинальная | Идентификатор |
| `boundary` | Географические границы кластера | Пространственная | Географический признак |
| `months` | Месяцы, к которым относится наблюдение | Временная | Временной признак |
| `prices_25` | 25-й перцентиль стоимости аренды | Отношений | Зависимая переменная |
| `prices_50` | Медианная стоимость аренды | Отношений | Основная зависимая переменная |
| `prices_75` | 75-й перцентиль стоимости аренды | Отношений | Зависимая переменная |
| `prices_50_meter` | Медианная стоимость аренды за квадратный метр | Отношений | Зависимая переменная |
| `nearest_metro_distance` | Расстояние до ближайшей станции метро | Отношений | Независимая переменная |
| `city_center_distance` | Расстояние до центра города | Отношений | Независимая переменная |
| `bus_stops` | Количество остановок общественного транспорта | Отношений | Независимая переменная |
| `property_listings_total` | Количество объявлений в кластере | Отношений | Контрольная переменная |
| `house_year_50` | Медианный год постройки зданий | Интервальная | Независимая переменная |
| `rooms_50` | Медианное количество комнат | Отношений | Независимая переменная |
| `renovation_type` | Наиболее распространённые типы ремонта в кластере | Номинальная | Независимая переменная |

Обозначения `_25`, `_50` и `_75` указывают на соответствующие перцентили распределения признака. Для исследования стоимости аренды основным показателем выступает `prices_50`, поскольку медиана позволяет оценивать типичный уровень цен и менее чувствительна к экстремальным значениям.


# Реляционный набор данных

In [22]:
df = pd.read_csv('russia_real_estate.csv')

In [23]:
moscow_rent = df.query(
    'city_name == "Moscow" and category == "flat rent"'
)

In [24]:
print('Количество строк', df.shape[0])
print('Количество столбцов', df.shape[1])

Количество строк 456182
Количество столбцов 59


In [25]:
print('Количество строк(Москва)', moscow_rent.shape[0])
print('Количество столбцов(Москва)', moscow_rent.shape[1])

Количество строк(Москва) 83473
Количество столбцов(Москва) 59


In [26]:
print(moscow_rent.info())

<class 'pandas.core.frame.DataFrame'>
Index: 83473 entries, 79735 to 163207
Data columns (total 59 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   cluster_id               83473 non-null  int64  
 1   city_name                83473 non-null  object 
 2   category                 83473 non-null  object 
 3   boundary                 83473 non-null  object 
 4   months                   83473 non-null  object 
 5   nearest_metro_distance   83473 non-null  float64
 6   city_center_distance     83473 non-null  float64
 7   bus_stops                83473 non-null  float64
 8   property_listings_total  83473 non-null  float64
 9   prices_25                83473 non-null  float64
 10  prices_50                83473 non-null  float64
 11  prices_75                83473 non-null  float64
 12  prices_1_room_25         77082 non-null  float64
 13  prices_1_room_50         77082 non-null  float64
 14  prices_1_room_75      

In [27]:
print('Количество пропусков:\n', moscow_rent.isna().sum())

Количество пропусков:
 cluster_id                     0
city_name                      0
category                       0
boundary                       0
months                         0
nearest_metro_distance         0
city_center_distance           0
bus_stops                      0
property_listings_total        0
prices_25                      0
prices_50                      0
prices_75                      0
prices_1_room_25            6391
prices_1_room_50            6391
prices_1_room_75            6391
prices_2_rooms_25           5242
prices_2_rooms_50           5242
prices_2_rooms_75           5242
prices_3_rooms_25          25805
prices_3_rooms_50          25805
prices_3_rooms_75          25805
prices_4_rooms_25          68617
prices_4_rooms_50          68617
prices_4_rooms_75          68617
prices_25_meter                0
prices_50_meter                0
prices_75_meter                0
prices_1_room_25_meter      6391
prices_1_room_50_meter      6391
prices_1_room_75_met

In [28]:
missing_percent = (moscow_rent.isna().mean() * 100).round(2)

print('Доля пропусков, %:')
display(missing_percent.to_frame("missing_percent"))

Доля пропусков, %:


,missing_percent
cluster_id,0.00
city_name,0.00
category,0.00
boundary,0.00
months,0.00
nearest_metro_distance,0.00
city_center_distance,0.00
bus_stops,0.00
property_listings_total,0.00
prices_25,0.00


In [29]:
print('Дубликаты:', moscow_rent.duplicated().sum())

Дубликаты: 0


In [30]:
display(moscow_rent.describe().T)

,count,mean,std,min,25%,50%,75%,max
cluster_id,83473.0,121471.000000,24096.723847,79735.000000,100603.000000,121471.000000,142339.000000,163207.000000
nearest_metro_distance,83473.0,922.868652,1032.877682,17.000000,480.000000,737.000000,1062.000000,33693.000000
city_center_distance,83473.0,13374.508763,7496.684957,517.000000,8507.000000,12630.000000,16321.000000,69158.000000
bus_stops,83473.0,3.908078,3.081207,0.000000,2.000000,3.000000,5.000000,36.000000
property_listings_total,83473.0,18.997065,27.114111,3.000000,6.000000,11.000000,21.000000,1381.000000
prices_25,83473.0,62076.286404,31134.511257,30000.000000,45750.000000,54750.000000,65500.000000,500000.000000
prices_50,83473.0,73951.066201,42557.190922,30000.000000,52500.000000,61500.000000,78500.000000,500000.000000
prices_75,83473.0,90380.461685,58124.628818,30000.000000,60000.000000,71250.000000,95000.000000,500000.000000
prices_1_room_25,77082.0,53863.289571,19536.494676,30000.000000,43000.000000,50000.000000,60000.000000,380000.000000
prices_1_room_50,77082.0,58245.400269,21981.894336,30000.000000,45000.000000,53000.000000,65000.000000,380000.000000


In [31]:
print(moscow_rent['months'].min())
print(moscow_rent['months'].max())

['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']
['2025-11', '2025-12']


In [32]:
moscow_rent.head(10)

,cluster_id,city_name,category,boundary,months,nearest_metro_distance,city_center_distance,bus_stops,property_listings_total,prices_25,...,lifetime_25,lifetime_50,lifetime_75,floor_25,floor_50,floor_75,total_floors_25,total_floors_50,total_floors_75,homeowner
79735,79735,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-01', '2025-02', '2025-03', '2025-04', '...",917.0,21601.0,2.0,81.0,54000.0,...,11.00,31.0,54.00,5.00,9.0,13.00,17.0,17.0,18.0,0.319444
79736,79736,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-07', '2025-08', '2025-09', '2025-10', '...",917.0,21601.0,2.0,38.0,55000.0,...,8.00,28.5,48.75,5.25,9.0,12.00,17.0,17.5,18.0,0.285714
79737,79737,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-05', '2025-06', '2025-07', '2025-08']",917.0,21601.0,2.0,33.0,52000.0,...,8.00,31.0,88.00,5.00,9.0,13.00,17.0,17.0,18.0,0.333333
79738,79738,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-07', '2025-08', '2025-09']",917.0,21601.0,2.0,22.0,50500.0,...,6.25,10.0,54.25,4.00,8.5,13.75,17.0,17.0,18.0,0.272727
79739,79739,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-07', '2025-08']",917.0,21601.0,2.0,19.0,51000.0,...,6.50,11.0,46.50,3.50,7.0,13.50,17.0,17.0,18.0,0.315789
79740,79740,Moscow,flat rent,"[((55.758587399999996, 37.5447924057971), (55....","['2025-01', '2025-02', '2025-03', '2025-04', '...",915.0,4415.0,0.0,90.0,100000.0,...,12.25,25.0,47.75,3.00,6.0,12.00,6.0,17.0,20.0,0.128571
79741,79741,Moscow,flat rent,"[((55.758587399999996, 37.5447924057971), (55....","['2025-01', '2025-02', '2025-03', '2025-04', '...",915.0,4415.0,0.0,48.0,97500.0,...,12.75,25.0,38.00,4.00,8.5,13.00,12.0,17.0,21.0,0.058824
79742,79742,Moscow,flat rent,"[((55.758587399999996, 37.5447924057971), (55....","['2025-01', '2025-02', '2025-03', '2025-04']",915.0,4415.0,0.0,37.0,100000.0,...,12.00,24.0,38.00,4.00,8.0,13.00,12.0,20.0,21.0,0.080000
79743,79743,Moscow,flat rent,"[((55.758587399999996, 37.5447924057971), (55....","['2025-01', '2025-02', '2025-03']",915.0,4415.0,0.0,31.0,95000.0,...,12.00,25.0,44.50,5.00,9.0,13.50,14.5,20.0,21.0,0.090909
79744,79744,Moscow,flat rent,"[((55.758587399999996, 37.5447924057971), (55....","['2025-01', '2025-02']",915.0,4415.0,0.0,22.0,129992.5,...,12.00,31.0,36.25,4.00,7.0,13.75,17.0,20.0,21.0,0.133333


Одна запись недвижимости может описывать период в несколько месяцев. Поэтому для объединения необходимо учитывать временной интервал каждой записи

In [33]:
print('Количество уникальных периодов:',moscow_rent['months'].nunique())

Количество уникальных периодов: 32


In [34]:
display(moscow_rent['months'].value_counts().sort_index())

months
['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']    3003
['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06']                                                                      2790
['2024-01', '2024-02', '2024-03', '2024-04']                                                                                            2563
['2024-01', '2024-02', '2024-03']                                                                                                       2381
['2024-01', '2024-02']                                                                                                                  2003
['2024-03', '2024-04']                                                                                                                  2282
['2024-04', '2024-05', '2024-06']                                                                                                       2563
['2024

In [35]:
rent_clean = moscow_rent.copy()

print("Размер таблицы:", rent_clean.shape)
print("Количество полных дубликатов:", rent_clean.duplicated().sum())

Размер таблицы: (83473, 59)
Количество полных дубликатов: 0


# Иерархических набор данных

In [4]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/AllRates-Today/central-bank-exchange-rates/main/data/cbr/history/"

cbr_2024 = pd.read_csv(base_url + "2024.csv")
cbr_2025 = pd.read_csv(base_url + "2025.csv")

cbr_raw = pd.concat([cbr_2024, cbr_2025], ignore_index=True)

cbr_raw["date"] = pd.to_datetime(cbr_raw["date"])

print("Размер датасета:", cbr_raw.shape)
display(cbr_raw.head())

Размер датасета: (22266, 5)


,date,base,quote,type,value
0,2024-01-10,AED,RUB,reference,24.616500
1,2024-01-10,AMD,RUB,reference,0.222884
2,2024-01-10,AUD,RUB,reference,60.742400
3,2024-01-10,AZN,RUB,reference,53.178800
4,2024-01-10,BRL,RUB,reference,18.507600


In [5]:
print("Период:", cbr_raw["date"].min(), "-", cbr_raw["date"].max())

print("\nТипы курсов:")
print(cbr_raw["type"].value_counts())

print("\nВалюты:")
print(sorted(cbr_raw["base"].unique()))
print(sorted(cbr_raw["quote"].unique()))

Период: 2024-01-10 00:00:00 - 2025-12-31 00:00:00

Типы курсов:
type
reference    22266
Name: count, dtype: int64

Валюты:
['AED', 'AMD', 'AUD', 'AZN', 'BDT', 'BHD', 'BOB', 'BRL', 'BYN', 'CAD', 'CHF', 'CNY', 'CUP', 'CZK', 'DKK', 'DZD', 'EGP', 'ETB', 'EUR', 'GBP', 'GEL', 'HKD', 'HUF', 'IDR', 'INR', 'IRR', 'JPY', 'KGS', 'KRW', 'KZT', 'MDL', 'MMK', 'MNT', 'NGN', 'NOK', 'NZD', 'OMR', 'PLN', 'QAR', 'RON', 'RSD', 'SAR', 'SEK', 'SGD', 'THB', 'TJS', 'TMT', 'TRY', 'UAH', 'USD', 'UZS', 'VND', 'XDR', 'ZAR']
['RUB']


In [6]:
print('Количество строк', cbr_raw.shape[0])
print('Количество столбцов', cbr_raw.shape[1])

Количество строк 22266
Количество столбцов 5


In [7]:
print(cbr_raw.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22266 entries, 0 to 22265
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    22266 non-null  datetime64[ns]
 1   base    22266 non-null  object        
 2   quote   22266 non-null  object        
 3   type    22266 non-null  object        
 4   value   22266 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 869.9+ KB
None


In [8]:
print('Количество пропусков:\n', cbr_raw.isna().sum())

Количество пропусков:
 date     0
base     0
quote    0
type     0
value    0
dtype: int64


In [9]:
print('Дубликаты:', cbr_raw.duplicated().sum())

Дубликаты: 0


In [10]:
currency_rates = cbr_raw[
    (cbr_raw["quote"] == "RUB") &
    (cbr_raw["base"].isin(["USD", "EUR"])) &
    (cbr_raw["type"] == "reference")
].copy()

currency_rates = currency_rates[
    (currency_rates["date"] >= "2024-01-01") &
    (currency_rates["date"] <= "2025-12-31")
].copy()

display(currency_rates.head())
print(currency_rates.groupby("base")["value"].agg(
    ["count", "min", "max", "mean", "std"]
))

,date,base,quote,type,value
12,2024-01-10,EUR,RUB,reference,98.9825
37,2024-01-10,USD,RUB,reference,90.4040
54,2024-01-11,EUR,RUB,reference,97.8745
79,2024-01-11,USD,RUB,reference,89.3939
96,2024-01-12,EUR,RUB,reference,97.5401


      count      min       max       mean       std
base                                               
EUR     495  87.5697  116.1410  97.172448  5.272971
USD     495  76.0937  109.5782  87.943323  7.357765


In [36]:
currency_rates["month"] = currency_rates["date"].dt.to_period("M")

cbr_monthly = (
    currency_rates
    .groupby(["base", "quote", "month"])
    .agg(
        exchange_rate_mean=("value", "mean")
    )
    .reset_index()
)

cbr_monthly["month"] = cbr_monthly["month"].astype(str)

display(cbr_monthly.head())
print("Размер:", cbr_monthly.shape)

,base,quote,month,exchange_rate_mean
0,EUR,RUB,2024-01,96.701150
1,EUR,RUB,2024-02,98.703510
2,EUR,RUB,2024-03,99.672586
3,EUR,RUB,2024-04,99.743648
4,EUR,RUB,2024-05,98.066889


Размер: (48, 4)


# Источник с API с выдачей в формате JSON

In [12]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 55.75, "longitude": 37.62,
    "start_date": "2020-01-01", "end_date": "2026-04-01",
    "daily": ["temperature_2m_mean", "precipitation_sum"],
    "timezone": "Europe/Moscow"
}
data = requests.get(url, params=params).json()

In [13]:
data

{'latitude': 55.782074,
 'longitude': 37.576374,
 'generationtime_ms': 1.0263919830322266,
 'utc_offset_seconds': 10800,
 'timezone': 'Europe/Moscow',
 'timezone_abbreviation': 'GMT+3',
 'elevation': 152.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_mean': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2020-01-01',
   '2020-01-02',
   '2020-01-03',
   '2020-01-04',
   '2020-01-05',
   '2020-01-06',
   '2020-01-07',
   '2020-01-08',
   '2020-01-09',
   '2020-01-10',
   '2020-01-11',
   '2020-01-12',
   '2020-01-13',
   '2020-01-14',
   '2020-01-15',
   '2020-01-16',
   '2020-01-17',
   '2020-01-18',
   '2020-01-19',
   '2020-01-20',
   '2020-01-21',
   '2020-01-22',
   '2020-01-23',
   '2020-01-24',
   '2020-01-25',
   '2020-01-26',
   '2020-01-27',
   '2020-01-28',
   '2020-01-29',
   '2020-01-30',
   '2020-01-31',
   '2020-02-01',
   '2020-02-02',
   '2020-02-03',
   '2020-02-04',
   '2020-02-05',
   '2020-02-06',
   '2020-02-07',
   '2020-02-08',
   '2020-02-09',

In [14]:
weather_daily = pd.DataFrame(data["daily"])

weather_daily = weather_daily.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_mean",
    "precipitation_sum": "precipitation"
})

weather_daily["date"] = pd.to_datetime(weather_daily["date"])

# Оставляем период, соответствующий основному датасету
weather_daily = weather_daily[
    (weather_daily["date"] >= "2024-01-01") &
    (weather_daily["date"] <= "2025-12-31")
].copy()

weather_daily["city_name"] = "Moscow"

print("Размер погодного датасета:", weather_daily.shape)
display(weather_daily.head())
display(weather_daily.describe())

Размер погодного датасета: (731, 4)


,date,temperature_mean,precipitation,city_name
1461,2024-01-01,-15.9,0.0,Moscow
1462,2024-01-02,-21.9,0.0,Moscow
1463,2024-01-03,-26.6,0.0,Moscow
1464,2024-01-04,-27.5,0.0,Moscow
1465,2024-01-05,-24.3,0.0,Moscow


,date,temperature_mean,precipitation
count,731,731.000000,731.000000
mean,2024-12-31 00:00:00,7.544323,1.752257
min,2024-01-01 00:00:00,-27.500000,0.000000
25%,2024-07-01 12:00:00,-0.300000,0.000000
50%,2024-12-31 00:00:00,7.000000,0.100000
75%,2025-07-01 12:00:00,16.900000,1.900000
max,2025-12-31 00:00:00,29.000000,33.000000
std,NaN,10.534306,3.555991


In [37]:
weather_daily["month"] = weather_daily["date"].dt.to_period("M")

weather_monthly = (
    weather_daily
    .groupby(["city_name", "month"])
    .agg(
        temperature_mean=("temperature_mean", "mean"),
        precipitation_sum=("precipitation", "sum")
    )
    .reset_index()
)

weather_monthly["month"] = weather_monthly["month"].astype(str)

display(weather_monthly.head())
print("Размер:", weather_monthly.shape)

,city_name,month,temperature_mean,precipitation_sum
0,Moscow,2024-01,-12.074194,42.5
1,Moscow,2024-02,-6.462069,44.7
2,Moscow,2024-03,-1.964516,8.8
3,Moscow,2024-04,9.690000,47.6
4,Moscow,2024-05,11.764516,33.4


Размер: (24, 4)


#  Соединение


### 5.1. Выбор структуры хранения

Для хранения подготовленных и объединённых данных выбран файловый формат **Apache Parquet**.

Данный формат подходит для аналитических задач благодаря колоночной организации данных, сжатию, сохранению типов столбцов и возможности считывания только необходимых атрибутов.

Рассматривались следующие варианты хранения:

- CSV — простой формат для обмена табличными данными, но менее эффективен при обработке больших объёмов;
- JSON — подходит для хранения иерархических структур и ответов API;
- SQLite — реляционная база данных, однако для текущего объёма проекта её использование не является необходимым;
- Parquet — оптимален для хранения и последующей аналитической обработки подготовленных таблиц.

Таким образом, для проекта выбрана файловая архитектура хранения с использованием Parquet.

Данные разделяются на три уровня:

1. `data/raw/` — исходные данные без изменений (CSV, JSON, XML).
2. `data/processed/` — очищенные и подготовленные таблицы недвижимости, погоды и валютных курсов.
3. `data/mart/` — итоговая объединённая таблица, предназначенная для аналитики и построения визуализаций.

Такое разделение позволяет сохранять исходные данные, повторно выполнять обработку и обновлять итоговую аналитическую витрину без необходимости заново организовывать весь проект.



### 5.2. Организация хранения и сохранение данных

Для хранения результатов обработки создаётся структура каталогов, разделяющая исходные, подготовленные и объединённые данные.


In [17]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from pathlib import Path

# Основной каталог проекта
data_dir = Path("data")

# Каталоги для разных этапов обработки
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
mart_dir = data_dir / "mart"

for folder in [raw_dir, processed_dir, mart_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Структура каталогов создана:")
print(raw_dir)
print(processed_dir)
print(mart_dir)

Структура каталогов создана:
data\raw
data\processed
data\mart


In [38]:
def save_table(df, path):
    df.to_parquet(path, index=False)
    print(f"Сохранено: {path}")


save_table(
    rent_clean,
    processed_dir / "real_estate_clean.parquet"
)

save_table(
    weather_monthly,
    processed_dir / "weather_monthly.parquet"
)

save_table(
    cbr_monthly,
    processed_dir / "cbr_monthly.parquet"
)

Сохранено: data\processed\real_estate_clean.parquet
Сохранено: data\processed\weather_monthly.parquet
Сохранено: data\processed\cbr_monthly.parquet


In [39]:
# Сформируем таблицу курсов валют: отдельные столбцы для USD и EUR
currency_rates["month"] = currency_rates["date"].dt.to_period("M").astype(str)

cbr_monthly = (
    currency_rates
    .groupby(["month", "base"])["value"]
    .mean()
    .unstack()
    .reset_index()
)

cbr_monthly = cbr_monthly.rename(columns={
    "USD": "usd_rub_mean",
    "EUR": "eur_rub_mean"
})

display(cbr_monthly.head())

base,month,eur_rub_mean,usd_rub_mean
0,2024-01,96.701150,88.680288
1,2024-02,98.703510,91.518255
2,2024-03,99.672586,91.731633
3,2024-04,99.743648,92.958567
4,2024-05,98.066889,90.607153


In [40]:
# Агрегируем погоду
weather_daily["month"] = weather_daily["date"].dt.to_period("M").astype(str)

weather_monthly = (
    weather_daily
    .groupby("month")
    .agg(
        temperature_mean=("temperature_mean", "mean"),
        precipitation_sum=("precipitation", "sum")
    )
    .reset_index()
)

display(weather_monthly.head())

,month,temperature_mean,precipitation_sum
0,2024-01,-12.074194,42.5
1,2024-02,-6.462069,44.7
2,2024-03,-1.964516,8.8
3,2024-04,9.690000,47.6
4,2024-05,11.764516,33.4


In [41]:
# Объединяем погодные и экономические данные
monthly_features = weather_monthly.merge(
    cbr_monthly,
    on="month",
    how="left"
)

display(monthly_features.head())
print("Количество месяцев:", len(monthly_features))

,month,temperature_mean,precipitation_sum,eur_rub_mean,usd_rub_mean
0,2024-01,-12.074194,42.5,96.701150,88.680288
1,2024-02,-6.462069,44.7,98.703510,91.518255
2,2024-03,-1.964516,8.8,99.672586,91.731633
3,2024-04,9.690000,47.6,99.743648,92.958567
4,2024-05,11.764516,33.4,98.066889,90.607153


Количество месяцев: 24


In [43]:
import ast

# Создаём индекс по месяцам
monthly_indexed = monthly_features.copy()
monthly_indexed["month"] = monthly_indexed["month"].astype(str)
monthly_indexed = monthly_indexed.set_index("month")


# Преобразуем строку со списком месяцев в настоящий список
def parse_months(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            value = [value]

    if value is None:
        return []

    if isinstance(value, (list, tuple)):
        return [str(pd.Period(m, freq="M")) for m in value]

    if pd.isna(value):
        return []

    return [str(pd.Period(value, freq="M"))]


# Рассчитываем внешние показатели за весь период
def get_period_features(months):
    months = parse_months(months)

    period_data = monthly_indexed.reindex(months)

    return pd.Series({
        "temperature_mean_period": period_data["temperature_mean"].mean(),
        "precipitation_mean_monthly": period_data["precipitation_sum"].mean(),
        "usd_rub_mean_period": period_data["usd_rub_mean"].mean(),
        "eur_rub_mean_period": period_data["eur_rub_mean"].mean()
    })


# Создаём итоговую таблицу
merged_data = rent_clean.copy()

# Добавляем внешние показатели
external_features = merged_data["months"].apply(get_period_features)

merged_data = pd.concat(
    [merged_data, external_features],
    axis=1
)

print("Размер итоговой таблицы:", merged_data.shape)
display(merged_data.head())

Размер итоговой таблицы: (83473, 63)


,cluster_id,city_name,category,boundary,months,nearest_metro_distance,city_center_distance,bus_stops,property_listings_total,prices_25,...,floor_50,floor_75,total_floors_25,total_floors_50,total_floors_75,homeowner,temperature_mean_period,precipitation_mean_monthly,usd_rub_mean_period,eur_rub_mean_period
79735,79735,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-01', '2025-02', '2025-03', '2025-04', '...",917.0,21601.0,2.0,81.0,54000.0,...,9.0,13.00,17.0,17.0,18.0,0.319444,8.057238,56.683333,83.584496,94.215884
79736,79736,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-07', '2025-08', '2025-09', '2025-10', '...",917.0,21601.0,2.0,38.0,55000.0,...,9.0,12.00,17.0,17.5,18.0,0.285714,10.206219,64.133333,80.256303,93.679481
79737,79737,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-05', '2025-06', '2025-07', '2025-08']",917.0,21601.0,2.0,33.0,52000.0,...,9.0,13.00,17.0,17.0,18.0,0.333333,17.440968,85.150000,79.443221,91.671443
79738,79738,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-07', '2025-08', '2025-09']",917.0,21601.0,2.0,22.0,50500.0,...,8.5,13.75,17.0,17.0,18.0,0.272727,17.567527,65.100000,80.623665,94.319923
79739,79739,Moscow,flat rent,"[((55.589697666666666, 37.41820237681159), (55...","['2025-07', '2025-08']",917.0,21601.0,2.0,19.0,51000.0,...,7.0,13.50,17.0,17.0,18.0,0.315789,19.561290,91.500000,79.450095,92.734525


In [44]:
print("Количество строк в исходной таблице:", len(rent_clean))
print("Количество строк после объединения:", len(merged_data))

print("\nПропущенные значения:")
display(merged_data[
    [
        "temperature_mean_period",
        "precipitation_mean_monthly",
        "usd_rub_mean_period",
        "eur_rub_mean_period"
    ]
].isna().sum())

Количество строк в исходной таблице: 83473
Количество строк после объединения: 83473

Пропущенные значения:


temperature_mean_period       0
precipitation_mean_monthly    0
usd_rub_mean_period           0
eur_rub_mean_period           0
dtype: int64